In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
import glob
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
from natsort import natsorted


import jax
import jax.numpy as jnp
import equinox as eqx


import matplotlib as mpl
from matplotlib import rc
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10 * 2.54})
mpl.rcParams['text.latex.preamble']=r"\usepackage{bm}\usepackage{amsmath}"

In [ ]:
from dmpe.utils.density_estimation import select_bandwidth, gaussian_kernel

In [ ]:
for dim in range(1, 5):
    a = jnp.arange(0.01, 1.0, 0.01)
    
    h = eqx.filter_vmap(select_bandwidth, in_axes=(None, None, None, 0))(2, dim, 100, a)
    plt.plot(a, h)
    plt.show()

### What is the impact of the bandwidth on the kernel and on JSD values? 

Evaluate in more depth:

In [ ]:
a = jnp.arange(0.1, 1.0, 0.1)

h = eqx.filter_vmap(select_bandwidth, in_axes=(None, None, None, 0))(2, dim, 100, a)
z = jnp.linspace(-1, 1, 1000)[..., None]
out = eqx.filter_vmap(gaussian_kernel, in_axes=(None, 0))(z, h)
out.shape

In [ ]:
for x in out:
    plt.plot(x)

plt.show()

### Load results plot metrics over bandwidth:

- test impact of bandwidth on the JSD (just use the generated data, no need for dummy data)

In [ ]:
import pathlib
import pandas as pd

from dmpe.data_management import DataPaths
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, evaluate_experiment_metrics
from dmpe.evaluation.experiment_utils import default_jsd, default_ae, default_mcudsa, default_ksfc, default_df

In [ ]:
from enum import Enum
class Systems(Enum):
    FLUID_TANK = 1
    PENDULUM = 2
    CART_POLE = 3

In [ ]:
sys_name = Systems.FLUID_TANK
algo_name = "dmpe"

In [ ]:
for system in Systems:
    print(system)

In [ ]:
"fluid_tank"

In [ ]:
result_path = DataPaths().sensitivity_analysis_experiments / pathlib.Path(algo_name) /  pathlib.Path(sys_name.name.lower())
exp_ids = get_experiment_ids(result_path)
exp_ids

In [ ]:
params, observations, actions, _ = load_experiment_results(exp_ids[0], result_path)

In [ ]:
a = jnp.arange(0.01, 1.0, 0.01)

points_per_dim = 50
dim = 1

h = eqx.filter_vmap(select_bandwidth, in_axes=(None, None, None, 0))(2, dim, points_per_dim, a)

jsd_values = eqx.filter_vmap(default_jsd, in_axes=(None, None, None, None, 0, None, None))(observations, actions, points_per_dim, (-1, 1), h, None, True)
plt.plot(jsd_values)

In [ ]:
a = jnp.arange(0.01, 1.0, 0.01)

points_per_dim = 50

h = jnp.arange(0.01, 5.0, 0.01)

jsd_values = eqx.filter_vmap(default_jsd, in_axes=(None, None, None, None, 0, None, None))(observations, actions, points_per_dim, (-1, 1), h, None, True)
plt.plot(jsd_values)

- place the metrics into a `pd.DataFrame` together with the a value and the bandwidth

### Move this into a script:

In [ ]:
sys_name = Systems.FLUID_TANK
algo_name = "dmpe"

In [ ]:
result_path = DataPaths().sensitivity_analysis_experiments / pathlib.Path(algo_name) /  pathlib.Path(sys_name.name.lower())
exp_ids = get_experiment_ids(result_path)
exp_ids

data_dict = {
    "exp_id": [],
    "a": [],
    "bandwidth": [],
    "seed": [],
    "jsd": [],
    "ae": [],
    "mcudsa": [],
    "ksfc": [],
    "df": [],
}

for exp_id in tqdm(exp_ids):
    params, observations, actions, _ = load_experiment_results(exp_id, result_path)
    data_dict["exp_id"].append(exp_id)
    data_dict["bandwidth"].append(params["alg_params"]["bandwidth"])
    data_dict["seed"].append(params["seed"])
    data_dict["a"].append(None)

    metrics = evaluate_experiment_metrics(
        observations,
        actions,
        metrics={
            "jsd": partial(default_jsd, points_per_dim=50, bandwidth=select_bandwidth(2, 3, 50, 0.3)),
            "ae": default_ae,
            "mcudsa": partial(default_mcudsa, points_per_dim=50),
            "ksfc": partial(default_ksfc, points_per_dim=50, eps=1e-6),
            "df": partial(default_df, points_per_dim=15),
        }
    )
    for metric_key, metric_value in metrics.items():
        data_dict[metric_key].append(metric_value)

results_df = pd.DataFrame(data_dict)

# store the df to disk:
results_df.to_pickle(f"sensitivity_analysis_data_{sys_name.name.lower()}_{algo_name}.pkl")

In [ ]:
results_df = pd.read_pickle(f"sensitivity_analysis_data_{sys_name.name.lower()}_{algo_name}.pkl")

In [ ]:
fig, axs = plt.subplots(5,1, figsize=(10, 8), sharex=True, constrained_layout=True)

for idx, metric_key in enumerate(["jsd", "ae", "mcudsa", "ksfc", "df"]):

    grouped = results_df.groupby("bandwidth")[metric_key].agg(["mean", "std"]).reset_index()

    axs[idx].plot(grouped["bandwidth"], grouped["mean"], marker="x")
    axs[idx].fill_between(
        grouped["bandwidth"],
        grouped["mean"] - grouped["std"],
        grouped["mean"] + grouped["std"],
        alpha=0.3,
    )
    axs[idx].set_ylabel(metric_key)

for ax in axs:
    ax.grid(alpha=0.5)
    ax.tick_params(which="major", axis="y", direction="in")
    ax.tick_params(which="both", axis="x", direction="in")
    ax.set_xscale("log")
    ax.set_yscale("log")

    
axs[-1].set_xlim(results_df["bandwidth"].min(), results_df["bandwidth"].max())
axs[-1].set_xlabel("$h$")

plt.show()

### visualize grid and kernel size:

In [ ]:
select_bandwidth(
    delta_z=2,
    dim=3,
    n_g=21,
    percentage=1e-5,
)

In [ ]:
points_per_dim=21

In [ ]:
1e-8 + 1e-8